# 손글씨 각장 표기 인식기 추가 학습 (PaddleOCR PP-OCRv6_medium_rec)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganggang-0605/vision-welding-ai/blob/main/vision/notebooks/colab_train_rec.ipynb)

1단계(시각 인식)가 쓰는 PaddleOCR 인식 모델을 **합성 손글씨**(F·V·S + 숫자)로 이어서 학습하고, **운영측 손글씨 7줄**로 학습 전·후를 비교합니다.

**준비**
1. 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`
2. Google Drive에 `hw_rec.zip`을 올려 두기 (기본 위치: `내 드라이브/pac_hw/hw_rec.zip`, 다르면 아래 설정에서 `DRIVE_DIR` 수정)
   - `hw_rec.zip`은 `vision/tools/synth_handwriting.py`로 만든 합성 데이터와 운영측 7줄(`pac_rec`) — 운영측 사진이 들어 있으니 공유 범위 주의

**순서** 위에서부터 셀을 차례로 실행. 학습은 T4에서 20~30분 예상.
학습 중 연결이 끊기면 처음부터 다시 실행하면 됩니다 — 체크포인트를 Drive에 저장해 두어 **이어서 학습**합니다.

**결과** `DRIVE_DIR/runs/PP-OCRv6_medium_rec/` 의 `inference.zip`(학습한 모델)과 `eval.json`(평가)을 받아 저장소 담당자에게 전달.

## 1. 설정

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/pac_hw"   # hw_rec.zip 을 올린 Drive 폴더
ZIP_NAME = "hw_rec.zip"
MODEL = "PP-OCRv6_medium_rec"                 # 1단계 기본 인식 모델과 같음 (vision/src/vision/ocr.py)
EPOCHS = 20
LEARNING_RATE = 1e-4                          # 사전 학습 모델을 이어서 학습하므로 기본값(5e-4)보다 낮게
BATCH = 64
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"  # nvidia-smi 의 CUDA 버전이 12.6 미만이면 cu118

SAVE_DIR = f"{DRIVE_DIR}/runs/{MODEL}"        # 체크포인트·결과 (Drive에 저장 → 끊겨도 남음)
DATA = "/content/data"                        # 압축을 푸는 곳 (Colab 로컬 디스크가 Drive보다 빠름)

## 2. GPU 확인, Drive 연결, 데이터 풀기

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print("Python", sys.version.split()[0])

In [ ]:
import os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

zip_path = f"{DRIVE_DIR}/{ZIP_NAME}"
assert os.path.exists(zip_path), f"{zip_path} 가 없습니다 — Drive 위치와 DRIVE_DIR 을 확인하세요"
if not os.path.exists(f"{DATA}/rec_train/labels.tsv"):
    os.makedirs(DATA, exist_ok=True)
    shutil.copy(zip_path, "/content/hw_rec.zip")
    zipfile.ZipFile("/content/hw_rec.zip").extractall(DATA)
for part in ["rec_train", "rec_val", "pac_rec"]:
    n = sum(1 for line in open(f"{DATA}/{part}/labels.tsv", encoding="utf-8") if line.strip())
    print(f"{part}: {n}줄")

## 3. 설치 (5~10분)
GPU용 PaddlePaddle 3.3.1(1단계와 같은 버전), PaddleOCR v3.7.0 학습 코드, 평가용 PaddleOCR 패키지, 각장 형식 후처리(저장소 `vision.marking`).
설치 뒤 `numpy` 버전 경고로 런타임 재시작을 요구하면 `런타임 → 세션 다시 시작` 후 **1번 설정 셀부터** 다시 실행하세요.

In [ ]:
!pip -q install paddlepaddle-gpu==3.3.1 -i {PADDLE_INDEX}
!test -d /content/PaddleOCR || git clone -q --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!pip -q install -r /content/PaddleOCR/requirements.txt
!pip -q install paddleocr==3.7.0
!test -d /content/vwa || git clone -q --depth 1 https://github.com/ganggang-0605/vision-welding-ai.git /content/vwa
import paddle
paddle.utils.run_check()

In [ ]:
PRETRAINED = f"/content/pretrained/{MODEL}_pretrained.pdparams"
if not os.path.exists(PRETRAINED):
    os.makedirs("/content/pretrained", exist_ok=True)
    !wget -q -O {PRETRAINED} https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/{MODEL}_pretrained.pdparams
print(PRETRAINED, os.path.getsize(PRETRAINED) // 1_000_000, "MB")

## 4. 학습 설정 만들기
PaddleOCR 기본 설정(`configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml`)에서 바꾸는 것:
- 학습 데이터 100% 사용 (기본은 에폭마다 50%), 에폭 수·학습률·배치
- 여러 줄 이어 붙이기 증강(RecConAug) 확률 0.5 → 0.1 — 우리 데이터는 표기 한 줄
- Colab CPU가 2개라 데이터 로딩 프로세스 2개
- 글자 사전은 그대로 (F·V·S·숫자·점이 모두 있고, 사전 학습 모델의 출력층을 그대로 씀)

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"/content/PaddleOCR/configs/rec/PP-OCRv6/{MODEL}.yml", encoding="utf-8"))
g = cfg["Global"]
g.update(epoch_num=EPOCHS, save_model_dir=SAVE_DIR, save_epoch_step=5, eval_batch_step=[0, 500], print_batch_step=50,
         pretrained_model=PRETRAINED, use_visualdl=False,
         character_dict_path=f"/content/PaddleOCR/{g['character_dict_path']}")
cfg["Optimizer"]["lr"].update(learning_rate=LEARNING_RATE, warmup_epoch=1)
train = cfg["Train"]
train["dataset"].update(data_dir=f"{DATA}/rec_train", label_file_list=[f"{DATA}/rec_train/labels.tsv"], ratio_list=[1.0])
for t in train["dataset"]["transforms"]:
    if "RecConAug" in t:
        t["RecConAug"]["prob"] = 0.1
train["sampler"]["first_bs"] = BATCH
train["loader"].update(batch_size_per_card=BATCH, num_workers=2)
cfg["Eval"]["dataset"].update(data_dir=f"{DATA}/rec_val", label_file_list=[f"{DATA}/rec_val/labels.tsv"])
cfg["Eval"]["loader"]["num_workers"] = 2
CONFIG = "/content/hw_rec.yml"
yaml.safe_dump(cfg, open(CONFIG, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
shutil.copy(CONFIG, f"{SAVE_DIR}/hw_rec.yml")
print(open(CONFIG, encoding="utf-8").read()[:800])

## 5. 학습 (T4에서 20~30분)
`acc`는 합성 검증 세트(학습에 안 쓴 글씨체) 완전 일치 비율. 끊기면 이 셀만 다시 실행 → Drive의 `latest`부터 이어서 학습.

In [ ]:
ckpt = f"{SAVE_DIR}/latest"
resume = f"-o Global.checkpoints={ckpt}" if os.path.exists(ckpt + ".pdparams") else ""
print("이어서 학습" if resume else "처음부터 학습")
!cd /content/PaddleOCR && python -u tools/train.py -c {CONFIG} {resume} 2>&1 | grep --line-buffered -v -i "warn"

## 6. 학습한 모델 내보내기

In [ ]:
EXPORT = f"{SAVE_DIR}/inference"
!cd /content/PaddleOCR && python tools/export_model.py -c {CONFIG} -o Global.pretrained_model={SAVE_DIR}/best_accuracy Global.save_inference_dir={EXPORT}
!ls -la {EXPORT}
shutil.make_archive(f"{SAVE_DIR}/inference", "zip", EXPORT)
print("저장:", f"{SAVE_DIR}/inference.zip")

## 7. 평가: 학습 전 vs 학습 후
- **운영측 손글씨 7줄** (사람이 그린 상자로 자른 줄, 정답 F5.5·V·F5.0·V5.5·F5.5·V·S6.5) — 실제 손글씨에 통하는지 보는 핵심 숫자. 학습 전은 0/7
- **합성 검증 500줄** (학습에 안 쓴 글씨체)
- `형식 후처리`: 저장소 `vision.marking.to_marking` — 첫 글자는 F·V·S, 나머지는 숫자로 바로잡음

In [ ]:
import json, cv2, numpy as np
sys.path.insert(0, "/content/vwa/vision/src")
from vision.marking import to_marking
from paddleocr import TextRecognition

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def load(part, limit=None):
    rows = [l.rstrip("\n").split("\t") for l in open(f"{DATA}/{part}/labels.tsv", encoding="utf-8") if l.strip()][:limit]
    return [(rel, cv2.imread(f"{DATA}/{part}/{rel}"), label) for rel, label in rows]

def score(preds, labels):
    chars = sum(len(l) for l in labels)
    return {"exact": round(sum(p == l for p, l in zip(preds, labels)) / len(labels), 4),
            "cer": round(sum(min(edit_distance(p, l), len(l)) for p, l in zip(preds, labels)) / chars, 4)}

sets = {"pac_rec": load("pac_rec"), "rec_val": load("rec_val", 500)}
models = {"학습 전": TextRecognition(model_name=MODEL), "학습 후": TextRecognition(model_name=MODEL, model_dir=EXPORT)}
report = {}
for name, model in models.items():
    report[name] = {}
    for part, rows in sets.items():
        out = [(r["rec_text"], float(r["rec_score"])) for r in model.predict([img for _, img, _ in rows], batch_size=32)]
        labels = [label for _, _, label in rows]
        raw = [t for t, _ in out]
        fmt = [to_marking(t) or "" for t in raw]
        report[name][part] = {"raw": score(raw, labels), "format": score(fmt, labels),
                              "details": [{"id": rel, "label": l, "pred": t, "prob": round(p, 4), "format": f}
                                          for (rel, _, l), (t, p), f in zip(rows, out, fmt)]}
        print(f"{name} · {part}: 그대로 {report[name][part]['raw']} → 형식 후처리 {report[name][part]['format']}")

print("\n운영측 7줄 (정답 → 학습 전 / 학습 후)")
for a, b in zip(report["학습 전"]["pac_rec"]["details"], report["학습 후"]["pac_rec"]["details"]):
    print(f"  {a['label']:6s} → {a['pred']!r:12s} / {b['pred']!r:12s} ({b['prob']:.2f}) 형식 {b['format']}")
json.dump(report, open(f"{SAVE_DIR}/eval.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\n저장:", f"{SAVE_DIR}/eval.json")

## 다음
- `inference.zip`과 `eval.json`을 저장소 담당자에게 전달 → 1단계 `vision/src/vision/ocr.py`에 학습한 모델 경로를 연결하고 원격으로 전체 평가(steel-ocr·MPSC에서 인쇄체 성능이 떨어지지 않았는지 포함)
- 손글씨 위치 찾기(검출기) 학습은 `hw_det.zip`으로 같은 방식의 노트북을 따로 만듦